# Finance Dashboard — Data Exploration & Validation

**Project:** MarketCast Finance Dashboard  
**Author:** Siham Chowdhury  
**Purpose:** Explore the Azure PostgreSQL data warehouse, understand table structures, and validate key metrics against the Excel source of truth before building the dashboard.

---

## Overview

The data warehouse lives at `mc-data-warehouse.postgres.database.azure.com`. It has four schemas relevant to this project:

| Schema | Description |
|---|---|
| `core` | Clean, analytics-ready layer built on top of NetSuite and HubSpot data. **This is what we use.** |
| `hubspot` | Raw HubSpot CRM tables synced via Fivetran |
| `netsuite_netsuite` | Raw NetSuite financial tables |
| `screendragon` | Timesheet and resource management tables |

The dashboard is built on **four tables** in the `core` schema:

| Table | What it contains |
|---|---|
| `core.rpt_project_revenue_and_costs` | Main fact table — every financial transaction line (revenue and cost) |
| `core.rpt_hubspot_line_report` | Pipeline data — HubSpot deal lines |
| `core.fact_hubspot_targets` | Quarterly revenue targets by person and team |
| `core.fact_sd_timesheet_cost` | People costs — daily timesheet entries per staff member per project |

---

## 1. Setup

Connect to the database. The `get_engine()` function reads credentials from `.streamlit/secrets.toml` when running locally via Streamlit, or from `.env` when running plain Python scripts. Either way it connects to the same Azure DB.

> **Note:** Always call `engine.dispose()` after any failed query. A broken transaction leaves the connection in a `PendingRollbackError` state — disposing resets the connection pool.

In [2]:
import pandas as pd
from src.db.connection import get_engine

engine = get_engine()
engine.dispose()  # always start fresh

print("Connected to DB")

Connected to DB


---
## 2. Schema Discovery

First thing we did was discover what tables exist in the DB and which schema they live in.

Key finding: all analytics-ready tables are in the `core` schema, not `public`. This is why the initial queries failed — we had to prefix all table names with `core.`.

In [30]:
# List every table across all schemas (excluding system schemas)
pd.read_sql("""
    SELECT table_schema, table_name 
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name
""", engine)

,table_schema,table_name
0,core,bridge_deal_line
1,core,dim_accounting_periods
2,core,dim_accounts
3,core,dim_classes
4,core,dim_consolidated_exchange_rates
...,...,...
198,screendragon,sd_budget_by_project
199,screendragon,sd_role_rate
200,screendragon,sd_timesheet
201,screendragon,sd_timesheet_stage


---
## 3. Main Fact Table — `core.rpt_project_revenue_and_costs`

This is the most important table. Every row is a **financial transaction line** — either a revenue entry or a cost entry — tied to a specific project, client, service line, vertical, and accounting period.

### Key columns

| Column | Type | Description |
|---|---|---|
| `amount_usd` | float | The monetary value in USD. **Revenue rows are negative** (standard accounting double-entry convention — credits are negative). Always negate when summing revenue. |
| `account_type_id` | varchar | `'Income'` for revenue rows, `'COGS'` for cost rows. This is the primary filter for separating revenue from cost. |
| `accounting_period_start_date` | date | First day of the accounting period (month). Use this for time-series grouping. |
| `accounting_period_is_posted` | bool | `TRUE` means the period is finalised and posted to the ledger. **Always filter to `TRUE`** to exclude draft entries. |
| `top_level_parent_customer_name` | varchar | The ultimate parent client name (e.g. Disney, not individual subsidiaries). Use this for client-level aggregation. |
| `service_line_name` | text | The service line (Ad Solutions, Brand Tracking, Custom, Content, Data Science). |
| `vertical_name` | varchar | The vertical/industry (Theatrical/Studio, Financial Services, Auto, etc.). |
| `project_number` | varchar | Unique project identifier (e.g. PRJ55238). Used for counting distinct projects. |
| `role_name_at_timesheet_date` | varchar | Staff role when timesheets are embedded in this table (e.g. for people cost classification). |

### Important: the `Internal` vertical
Some rows have `vertical_name = 'Internal'` with costs but zero revenue. These are internal recharges and are **excluded from the dashboard** to avoid skewing profitability numbers. Needs confirmation from John Benak.

In [ ]:
# Full column schema for the main fact table
pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core' AND table_name = 'dim_customers'
    ORDER BY ordinal_position
""", engine)

In [5]:
# Sample 10 rows — notice revenue rows have negative amount_usd
# and account_type_id shows the row type (Income vs COGS)
pd.read_sql("""
    SELECT DISTINCT account_type_id, COUNT(*) as n
    FROM core.rpt_project_revenue_and_costs
    GROUP BY 1
    ORDER BY 2 DESC
""", engine)

,account_type_id,n
0,NaN,1482726
1,COGS,292509
2,Income,55793


In [6]:
# Full column schema for the main fact table
pd.read_sql("""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_schema = 'core' AND table_name = 'rpt_project_revenue_and_costs'
    ORDER BY ordinal_position
""", engine)

,column_name,data_type
0,transaction_id,bigint
1,transaction_number,character varying
2,transaction_type,character varying
3,transaction_line_unique_key,bigint
4,transaction_document_number,character varying
5,start_date,timestamp with time zone
6,end_date,timestamp with time zone
7,project_id,bigint
8,project_number,character varying
9,project_name,character varying


In [7]:
# Sample 10 rows — notice revenue rows have negative amount_usd
# and account_type_id shows the row type (Income vs COGS)
pd.read_sql("SELECT * FROM core.rpt_project_revenue_and_costs LIMIT 10", engine)

,transaction_id,transaction_number,transaction_type,transaction_line_unique_key,transaction_document_number,start_date,end_date,project_id,project_number,project_name,...,amount,amount_usd,hours,timesheet_full_name,role_name_at_timesheet_date,is_part_of_income_statement,source_query,originating_so_line_unique_key,methodology_id,methodology_name
0,670205,JOURNAL9498,Journal,3956021,Journal #Historical_Deductive Limited_TB_Feb2021,None,None,NaN,NaN,NaN,...,16259.95,21875.384545,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
1,989610,JOURNAL17416,Journal,5144384,Journal #JE17203,None,None,NaN,NaN,NaN,...,-928759.14,-928759.140000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
2,1074380,JOURNAL18729,Journal,5370598,Journal #JE18516,None,None,141788.0,PRJ55238,WBD Max UK-DE-IT Positioning,...,3360.00,4231.248000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
3,1074374,JOURNAL18728,Journal,5370548,Journal #JE18515,None,None,141711.0,PRJ55177,Mufasa: The Lion King Exits #1 UK ES IT JP KR ...,...,37.25,46.908925,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
4,1075665,JOURNAL18755,Journal,5375154,Journal #JE18542,None,None,141992.0,PRJ55414,Captain America: Brave New World UK ES IT MX B...,...,60.00,75.558000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
5,1074380,JOURNAL18729,Journal,5370604,Journal #JE18516,None,None,133370.0,PRJ53079,Audible BEAT Tracker 23-26,...,137.76,173.481168,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
6,1074380,JOURNAL18729,Journal,5370602,Journal #JE18516,None,None,141643.0,PRJ55124,Ford 2025 Global BCX Program,...,210.00,264.453000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
7,1074381,JOURNAL18730,Journal,5370642,Journal #JE18517,None,None,141769.0,PRJ55224,Universal Pictures - WICKED global post-releas...,...,170.00,214.081000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
8,1074346,JOURNAL18727,Journal,5370320,Journal #JE18514,None,None,141724.0,PRJ55184,LEGO P11 Pricing (US/DE) 2024,...,5117.87,6444.933691,None,None,None,True,qry_vendor_bills_and_credits,None,None,None
9,1075665,JOURNAL18755,Journal,5375170,Journal #JE18542,None,None,141910.0,PRJ55337,NBA - EU Research - Qual Only,...,80.00,100.744000,None,None,None,True,qry_vendor_bills_and_credits,None,None,None


---
## 4. Pipeline Table — `core.rpt_hubspot_line_report`

This table contains **HubSpot CRM deal lines** — each row is a deal or deal line with its current pipeline stage, value, owner, and dimensions.

### Key columns

| Column | Type | Description |
|---|---|---|
| `deal_id` | bigint | Unique identifier for the deal. One deal can have multiple rows (one per line item). |
| `deal_pipeline_stage_name` | varchar | Current stage in the pipeline (Stage 1 through Stage 5, Closed Won, Closed Lost). Pipeline queries exclude Closed Won and Closed Lost. |
| `deal_amount_usd` | float | Total deal value in USD. **This is the correct value column** — `coalesced_amount_usd` and `line_amount_usd` were found to be null for most active pipeline deals. |
| `service_line` | varchar | Service line for the deal. **Watch out:** this column sometimes contains semicolon-separated values like `"Brand Tracking;Campaign Analytics"` when a deal spans multiple service lines. Requires an explode operation in Python. |
| `is_deal_deleted` | bool | Always filter to `FALSE` to exclude soft-deleted deals. |
| `owner_full_name` | text | The deal owner's full name. |
| `vertical` | varchar | The vertical for the deal. |

### The semicolon problem
HubSpot allows tagging a deal with multiple service lines. When this happens, the `service_line` column contains a semicolon-separated string. The pipeline query pulls individual deal lines (not pre-grouped), then Python uses `str.split(";")` followed by `explode()` to turn each service line into its own row. This is handled in `src/models/financials.py` in `get_pipeline()`.

In [8]:
# Sample the pipeline table
pd.read_sql("SELECT * FROM core.rpt_hubspot_line_report LIMIT 10", engine)

,deal_line_id,deal_id,deal_name,deal_pipeline_id,deal_pipeline_stage_id,deal_pipeline_stage_name,project_id,netsuite_customer_name,hubspot_service_name,line_amount,...,vertical,sub_vertical_name,industry,sub_industry,service_line,product,deal_type,is_reccuring_baseline_revs,closed_lost_reason,is_addendum
0,4.048281e+10,39967275541,L'Oreal NPI 2026,default,76355655,Closed Won,143533.0,"LOreal USA, Inc.",1-New Deal Creation Product Unknown,17000.0,...,FHAB,FHAB - CPG/HC,None,None,Campaign Analytics,547,Annual Renewal,True,NaN,159.0
1,4.493259e+10,39967275541,L'Oreal NPI 2026,default,76355655,Closed Won,143533.0,"LOreal USA, Inc.",AS - BE - Linear,20000.0,...,FHAB,FHAB - CPG/HC,None,None,Campaign Analytics,547,Annual Renewal,True,NaN,159.0
2,4.951678e+10,54101283728,Charter Spectrum Q1 2026 AE Copy Test,default,76355655,Closed Won,143604.0,Charter Communications,AS - AE - AE Premium,13200.0,...,171,TTP - Telco & Cable,None,None,Campaign Analytics,NaN,Annual Renewal,False,NaN,159.0
3,5.196487e+10,56697985097,T-Mobile Sponsorship Renewal 2026 - LVGP Onlin...,default,76355655,Closed Won,143784.0,T-Mobile,BS - SI - Sponsorship Impact,85000.0,...,171,TTP - Telco & Cable,None,None,Brand Tracking,NaN,Annual Renewal,None,NaN,NaN
4,5.197798e+10,56697985097,T-Mobile Sponsorship Renewal 2026 - LVGP Onlin...,default,76355655,Closed Won,143784.0,T-Mobile,BS - SI - Sponsorship Impact,209500.0,...,171,TTP - Telco & Cable,None,None,Brand Tracking,NaN,Annual Renewal,None,NaN,NaN
5,5.197799e+10,56697985097,T-Mobile Sponsorship Renewal 2026 - LVGP Onlin...,default,76355655,Closed Won,143784.0,T-Mobile,BS - SI - Sponsorship Impact,78500.0,...,171,TTP - Telco & Cable,None,None,Brand Tracking,NaN,Annual Renewal,None,NaN,NaN
6,NaN,58295134258,Chick-fil-A SLTV day part expansion BE,default,closedlost,Closed lost,NaN,NaN,NaN,NaN,...,FHAB,FHAB - Brands,None,None,Campaign Analytics,NaN,NBC,False,Price too High,NaN
7,4.994481e+10,54610037383,Liquid Death 2026 Super Bowl,default,closedlost,Closed lost,NaN,NaN,AS - BE - BE Super Bowl,50000.0,...,FHAB,FHAB - CPG/HC,None,None,Campaign Analytics,603,newbusiness,False,Client Went Silent,NaN
8,NaN,42556911396,sara testing 20250820,default,contractsent,Stage 2 - Solutioning,NaN,NaN,NaN,NaN,...,167,MSE - Sports & Live Events,None,None,NaN,NaN,NaN,False,NaN,NaN
9,5.110763e+10,58630799288,Walmart Apr-May26 AEE,default,19129243,Stage 4 - Negotiating / Closing,NaN,NaN,AS - AE - AE Express (AEE),78000.0,...,FHAB,FHAB - Brands,None,None,Campaign Analytics,NaN,existingbusiness,False,NaN,159.0


In [9]:
# ── Demonstrate the explode cases ─────────────────────────────

df_500 = pd.read_sql("SELECT * FROM core.rpt_hubspot_line_report LIMIT 500", engine)

# Case 1: rows where service_line contains a semicolon
# These are deals tagged with multiple service lines in HubSpot
# After explode, each service line becomes its own row
case1 = df_500[df_500["service_line"].str.contains(";", na=False)]
print(f"=== Case 1: Multi service line deals ({len(case1)} rows) ===")
print("These rows will be SPLIT by explode — one row per service line")
print(case1[["deal_id", "deal_pipeline_stage_name", "service_line", "deal_amount_usd"]].head(5))

# Case 2: rows where service_line has NO semicolon
# These are normal single-service-line deals — explode has no effect
case2 = df_500[
    ~df_500["service_line"].str.contains(";", na=False) &
    df_500["service_line"].notna()
]
print(f"\n=== Case 2: Single service line deals ({len(case2)} rows) ===")
print("These rows are UNCHANGED by explode")
print(case2[["deal_id", "deal_pipeline_stage_name", "service_line", "deal_amount_usd"]].head(5))

# Now show what happens after explode on Case 1
print("\n=== After explode on Case 1 ===")
if len(case1) > 0:
    example = case1.head(3).copy()
    print(f"Before: {len(example)} rows")
    print(example[["deal_id", "service_line", "deal_amount_usd"]])

    example["service_line"] = example["service_line"].str.split(";")
    example = example.explode("service_line")
    example["service_line"] = example["service_line"].str.strip()
    print(f"\nAfter: {len(example)} rows — same deal_id, split across service lines")
    print(example[["deal_id", "service_line", "deal_amount_usd"]])
else:
    print("No semicolon cases in this 500-row sample — try a larger sample")

=== Case 1: Multi service line deals (18 rows) ===
These rows will be SPLIT by explode — one row per service line
        deal_id deal_pipeline_stage_name                     service_line  \
16  45096483952              Closed lost  Data Science;Campaign Analytics   
23  52276104841               Closed Won          Custom Research;Content   
24  52276104841               Closed Won          Custom Research;Content   
41  58634154640              Closed lost       Campaign Analytics;Content   
42  58634154640              Closed lost       Campaign Analytics;Content   

    deal_amount_usd  
16        1000000.0  
23          45200.0  
24          45200.0  
41          18000.0  
42          18000.0  

=== Case 2: Single service line deals (422 rows) ===
These rows are UNCHANGED by explode
       deal_id deal_pipeline_stage_name        service_line  deal_amount_usd
0  39967275541               Closed Won  Campaign Analytics          20000.0
1  39967275541               Closed Won  Campai

---
## 5. Targets Table — `core.fact_hubspot_targets`

Quarterly revenue targets by person and team. Simple table.

### Key limitation
This table only contains data from **Q1 2025 onwards**. There is no 2024 target history in the DB. The 2026 budget lives in the Excel file (`Budget` and `Budget SL` sheets) and has not been loaded into the DB. This means Actual vs Budget cannot be built without resolving this gap with Paul/Erik.

In [10]:
# Sample targets table — note earliest date is Q1 2025
pd.read_sql("SELECT * FROM core.fact_hubspot_targets LIMIT 10", engine)

,_line,_fivetran_synced,team_primary_name,user_first_name,user_last_name,target_amount_usd,quarter_start_date,quarter_end_date
0,3,2025-06-23 21:08:19.052000+00:00,FSA,Mark,Willard,5300000,2025-10-01,2025-12-31
1,2,2025-06-23 21:08:19.052000+00:00,FSA,Mike,Chung,1100000,2025-07-01,2025-09-30
2,39,2025-06-23 21:08:19.056000+00:00,Brands - CPG,Duncan,Stark,1300000,2025-07-01,2025-09-30
3,17,2025-06-23 21:08:19.053000+00:00,Brands - CPG,Juliann,Pavlovcic,2700000,2025-01-01,2025-03-31
4,69,2025-06-23 21:08:19.058000+00:00,Europe,Christian,Dubreuil,1800000,2025-01-01,2025-03-31
5,47,2025-06-23 21:08:19.056000+00:00,FSA,Mark,Cosby,800000,2025-04-01,2025-06-30
6,75,2025-06-23 21:08:19.059000+00:00,MSE,Julanne,Schiffer,3600000,2025-04-01,2025-06-30
7,41,2025-06-23 21:08:19.056000+00:00,FSA,Mark,Sutin,0,2025-07-01,2025-09-30
8,71,2025-06-23 21:08:19.058000+00:00,MSE,David,Halas,800000,2025-07-01,2025-09-30
9,67,2025-06-23 21:08:19.058000+00:00,MSE,Dounia,Turrill,800000,2025-10-01,2025-12-31


---
## 6. People Costs Table — `core.fact_sd_timesheet_cost`

Daily timesheet entries from Screendragon — the resource management tool. Each row is one person's hours on one project on one day.

### Key columns

| Column | Type | Description |
|---|---|---|
| `netsuite_project_id` | bigint | Joins to `project_id` in `rpt_project_revenue_and_costs`. |
| `accounting_period_id` | bigint | Joins to `accounting_period_id` in the main table. |
| `timesheet_external_cost` | float | Cost at the external (billable) rate — **this is the people cost figure for profitability**. |
| `timesheet_internal_cost` | float | Cost at the internal rate — lower than external, used for internal reporting. |
| `actual_hours` | float | Hours logged for that day. Zero-hour rows exist (staff on a project but no hours logged that day). |
| `role_name_at_timesheet_date` | varchar | The person's role at the time of the timesheet (e.g. `Ops - Field Management (MK)`, `VP Research`). Used to classify labor type (Ops vs Research). |

### The join inflation problem
Joining `fact_sd_timesheet_cost` directly to `rpt_project_revenue_and_costs` inflates people costs massively because `rpt_project_revenue_and_costs` has **many transaction lines per project per period** (one per vendor bill, one per revenue recognition entry etc.). A naive join multiplies each timesheet cost row by the number of transaction lines for that project/period combination.

**The fix:** use a CTE to pre-aggregate timesheet costs to `(project, period)` grain **before** joining. See `profitability.sql` for the implementation.

### Current status
Even with the CTE fix, people costs ($68M for 2025) appear higher than COS ($31M), which is unusual for a research/services company. Needs validation with John Benak — the correct total and the correct table/columns to use need to be confirmed.

In [11]:
# Full column schema
pd.read_sql("""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_schema = 'core' AND table_name = 'fact_sd_timesheet_cost'
    ORDER BY ordinal_position
""", engine)

,column_name,data_type
0,netsuite_project_id,bigint
1,sd_project_id,character varying
2,netsuite_project_number,character varying
3,user_id,integer
4,category,character varying
5,actual_hours,double precision
6,timesheet_date,timestamp without time zone
7,user_first_name,character varying
8,user_last_name,character varying
9,role_name_at_timesheet_date,character varying


In [12]:
# Sample — note rows 0-5 have 0.0 actual_hours but still have an external rate
# These are zero-cost days (staff on a project but not logging hours)
pd.read_sql("SELECT * FROM core.fact_sd_timesheet_cost LIMIT 10", engine)

,netsuite_project_id,sd_project_id,netsuite_project_number,user_id,category,actual_hours,timesheet_date,user_first_name,user_last_name,role_name_at_timesheet_date,...,external_rate_at_timesheet_date,timesheet_external_cost,internal_rate_at_timesheet_date,timesheet_internal_cost,accounting_period_id,accounting_period_start_date,accounting_period_ending_date,accounting_period_closed_at,accounting_period_is_posted,accounting_period_is_closed
0,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-25,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
1,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-26,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
2,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-27,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
3,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-28,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
4,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-29,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
5,133238,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-30,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
6,133238,PRJ53001,PRJ53001,1061,Billable - Operations,4.4,2024-12-01,A,Jatin,Ops - Field Management (MK),...,137,602.8,47,206.8,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True
7,134645,PRJ53516,PRJ53516,1061,Billable - Operations,1.1,2024-12-02,A,Jatin,Ops - Field Management (MK),...,137,150.7,47,51.7,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True
8,134645,PRJ53516,PRJ53516,1061,Billable - Operations,0.0,2024-12-03,A,Jatin,Ops - Field Management (MK),...,137,0.0,47,0.0,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True
9,134645,PRJ53516,PRJ53516,1061,Billable - Operations,2.2,2024-12-04,A,Jatin,Ops - Field Management (MK),...,137,301.4,47,103.4,136,2024-12-01,2024-12-31,2026-01-28 10:23:53+00:00,True,True


---
## 7. Validation Against the Excel Source of Truth

The Excel file (`Revenue Data Warehouse - back up v2-2.xlsx`) is the finance team's existing reporting workbook. It is built on top of the same Azure DB using Power Pivot (Windows only). The key validation checks compare SQL query results against Excel pivot table outputs.

### Revenue logic
Revenue in the DB is stored as **negative** (standard double-entry accounting — revenue credits are negative). The DAX formula in Power Pivot confirms:
```
Revenue = -CALCULATE(SUM(rpt_project_revenue_and_costs[amount]))
```
Our SQL equivalent:
```sql
-SUM(amount_usd) WHERE account_type_id = 'Income' AND accounting_period_is_posted = TRUE
```

In [13]:
# Total 2025 revenue from the DB
# Expected: ~$135.7M based on Excel
df = pd.read_sql("""
    SELECT -SUM(amount_usd) AS total_revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)
print(f"SQL 2025 Revenue: ${df['total_revenue_2025'][0]/1e6:.1f}M")

SQL 2025 Revenue: $135.7M


In [14]:
# Total 2025 revenue from the Excel Flash - SL sheet
# Row 17 = Grand Total, columns 4-15 = months Jan-Dec 2025
# Excel values are in $000s so multiply by 1000 to compare
xl = pd.read_excel('../Revenue Data Warehouse - back up v2-2.xlsx', 
                   sheet_name='Flash - SL', header=None)

excel_2025_total = xl.iloc[17].iloc[4:16].sum()
print(f"Excel 2025 Revenue: ${excel_2025_total/1000:.1f}M (in $000s: ${excel_2025_total:,.0f}k)")

Excel 2025 Revenue: $135.7M (in $000s: $135,696k)


### Result: Revenue — ✅ VERIFIED
SQL: **$135.7M** | Excel: **$135.7M** — exact match to the penny.

This confirms:
- The `account_type_id = 'Income'` filter is correct
- Negating `amount_usd` is correct
- `accounting_period_is_posted = TRUE` is the right filter
- Our DB connection is hitting the same source as the Excel Power Pivot model

In [15]:
# Revenue by service line — SQL vs Excel comparison
# Excel values are in $000s; SQL values are in dollars
# Ad Solutions $83.1M SQL = 83,119k / 1000 = $83.1M Excel ✅

sql_sl = pd.read_sql("""
    SELECT 
        COALESCE(service_line_name, '(blank)') as service_line,
        -SUM(amount_usd) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 2 DESC
""", engine)

excel_sl = xl.iloc[5:17].copy().reset_index(drop=True)
excel_sl['service_line'] = excel_sl[2].ffill()
excel_sl['revenue_2025_k'] = excel_sl.iloc[:, 4:16].sum(axis=1)
excel_sl = excel_sl[['service_line', 'revenue_2025_k']].dropna()

print("=== SQL (in $) ===")
print(sql_sl.to_string(index=False))
print("\n=== Excel (in $000s) ===")
print(excel_sl.to_string(index=False))

=== SQL (in $) ===
  service_line  revenue_2025
  Ad Solutions   83118879.11
        Custom   24618215.50
Brand Tracking   23498047.49
       Content    3133214.10
  Data Science    2161222.08
       (blank)    -834067.93

=== Excel (in $000s) ===
  service_line revenue_2025_k
  Ad Solutions    17687.06112
  Ad Solutions     3225.53275
  Ad Solutions     2852.03826
  Ad Solutions    22873.19865
  Ad Solutions    30604.37283
  Ad Solutions      5876.6755
Brand Tracking    21103.45883
Brand Tracking     2394.58866
        Custom     24618.2155
       Content      3133.2141
  Data Science     2161.22208
       (blank)     -834.06793


### Result: Revenue by Service Line — ✅ VERIFIED
After adjusting for the $000s unit difference in Excel:

| Service Line | SQL ($M) | Excel ($M) | Match |
|---|---|---|---|
| Ad Solutions | $83.1M | $83.1M | ✅ |
| Custom | $24.6M | $24.6M | ✅ |
| Brand Tracking | $23.5M | $23.5M | ✅ |
| Content | $3.1M | $3.1M | ✅ |
| Data Science | $2.2M | $2.2M | ✅ |

In [16]:
# Top 10 clients by 2025 revenue
sql_clients = pd.read_sql("""
    SELECT 
        top_level_parent_customer_name as client,
        -SUM(amount_usd) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    AND top_level_parent_customer_name IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""", engine)

print(sql_clients.to_string(index=False))

          client  revenue_2025
    NBCUniversal   16297211.87
          Disney   12481249.04
            Meta    9400636.75
 Ford Motor Corp    7216523.75
            LEGO    6533258.74
          Amazon    5311251.11
          Google    4136333.29
Paramount Global    3931904.25
         Walmart    3393090.58
           Chase    3387885.54


In [17]:
# Spot check specific clients against the Excel Top Clients board deck sheet
# Excel shows Apple+ $2,019k, Fox $1,915k, Fidelity $1,482k etc.
# Multiply by 1000 to compare against SQL

sql_check = pd.read_sql("""
    SELECT 
        top_level_parent_customer_name as client,
        -SUM(amount_usd) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    AND top_level_parent_customer_name IN (
        'Apple+', 'Fox', 'Fidelity Investments', 
        'Home Depot', 'NBCUniversal', 'Disney'
    )
    GROUP BY 1
    ORDER BY 2 DESC
""", engine)

print(sql_check.to_string(index=False))

              client  revenue_2025
        NBCUniversal   16297211.87
              Disney   12481249.04
              Apple+    2019073.40
                 Fox    1914744.07
Fidelity Investments    1482406.66
          Home Depot    1154311.54


### Result: Top Clients — ✅ VERIFIED
Client values match the Excel exactly:
- Apple+: SQL $2,019,073 = Excel $2,019k ✅
- Fox: SQL $1,914,744 = Excel $1,915k ✅
- Fidelity: SQL $1,482,407 = Excel $1,482k ✅
- Home Depot: SQL $1,154,312 = Excel $1,154k ✅

The Excel `Top clients board deck` sheet was filtered to a subset — NBCUniversal and Disney don't appear there because they're on a different filtered view, but their values also match when cross-checked.

---
## 8. People Costs Join — The Fan-Out Problem

### What went wrong
The first attempt at joining `fact_sd_timesheet_cost` to `rpt_project_revenue_and_costs` produced **$20 billion** in people costs for 2024 — obviously wrong.

### Why it happened
`rpt_project_revenue_and_costs` has **many rows per project per period** (one per transaction line — vendor bills, revenue recognition entries, etc.). A direct join means each timesheet cost row gets multiplied by the number of transaction lines for that project/period. For a project with 50 transaction lines, the people cost gets counted 50 times.

### The fix — CTE pre-aggregation
Aggregate timesheet costs to `(project, period)` grain in a CTE **before** joining. This ensures one people cost row per project per period, matching the join key grain.

See `profitability.sql` for the full implementation using three CTEs:
1. `revenue_and_cos` — aggregates revenue and COS from the main table at project/period level
2. `people` — aggregates timesheet costs at project/period level  
3. `joined` — left joins people costs onto revenue/COS, then final SELECT groups up to service line/vertical

In [18]:
# Demonstrating the fan-out problem — naive join gives $20B people cost
# DO NOT USE this in production — for illustration only
pd.read_sql("""
    WITH people_costs AS (
        SELECT
            netsuite_project_id,
            accounting_period_id,
            SUM(timesheet_external_cost) AS people_cost_usd
        FROM core.fact_sd_timesheet_cost
        WHERE accounting_period_is_posted = TRUE
        GROUP BY 1,2
    )
    SELECT 
        -SUM(CASE WHEN r.account_type_id = 'Income' THEN r.amount_usd ELSE 0 END) AS revenue_usd,
         SUM(CASE WHEN r.account_type_id = 'COGS'   THEN r.amount_usd ELSE 0 END) AS cos_usd,
        COALESCE(SUM(p.people_cost_usd), 0) AS people_cost_usd  -- still inflated!
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN people_costs p
        ON  r.project_id = p.netsuite_project_id
        AND r.accounting_period_id = p.accounting_period_id
    WHERE r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2024
""", engine)

,revenue_usd,cos_usd,people_cost_usd
0,1.552284e+08,4.805479e+07,2.026131e+10


---
## 9. Pipeline — The Explode Operation

### The semicolon problem
HubSpot allows tagging a deal with multiple service lines. When this happens, the `service_line` column in `rpt_hubspot_line_report` contains a semicolon-separated string like `"Brand Tracking;Campaign Analytics;Custom Research"`.

### Two distinct cases

**Case 1 — One deal, multiple service lines (semicolons in the string)**  
A single deal tagged with multiple service lines. After exploding, each service line gets its own row with the same deal value.

**Case 2 — Multiple deals, one service line (no semicolons, `num_deals > 1`)**  
Multiple separate deals that share the same stage + vertical + service line + owner combination, grouped together by the SQL.

### The pipeline SQL approach
The current pipeline query (`pipeline.sql`) pulls individual deal lines without pre-grouping, then all grouping and counting happens in Python after the explode. This avoids the ambiguity of what to do with `num_deals` when a group has both semicolons and multiple deals.

In [19]:
from pathlib import Path

# Load raw pipeline (before explode)
# Path uses .. to go up from notebooks/ to project root
sql = (Path("../src/queries") / "pipeline.sql").read_text()
df_raw = pd.read_sql(sql, engine)

print(f"Total rows before explode: {len(df_raw)}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(5)

FileNotFoundError: [Errno 2] No such file or directory: '../src/queries/pipeline.sql'

In [ ]:
# Show the explode operation step by step

# Step 1: find rows with semicolons (multi service line deals)
multi_sl = df_raw[df_raw["service_line"].str.contains(";", na=False)]
print(f"Rows with semicolons (multi service line): {len(multi_sl)}")
print(multi_sl[["deal_id", "deal_pipeline_stage_name", "service_line", "pipeline_value_usd"]].head(5))

In [ ]:
# Step 2: apply the explode
df_exploded = df_raw.copy()
df_exploded["service_line"] = df_exploded["service_line"].str.split(";")
df_exploded = df_exploded.explode("service_line")
df_exploded["service_line"] = df_exploded["service_line"].str.strip()

print(f"Rows after explode: {len(df_exploded)} (was {len(df_raw)})")
print(f"Extra rows created by explode: {len(df_exploded) - len(df_raw)}")

# Step 3: group correctly — count distinct deals per group
grouped = (
    df_exploded
    .groupby(["deal_pipeline_stage_name", "vertical", "service_line", "owner_full_name"])
    .agg(
        num_deals=("deal_id", "nunique"),
        pipeline_value_usd=("pipeline_value_usd", "sum")
    )
    .reset_index()
    .sort_values("pipeline_value_usd", ascending=False)
)

print(f"\nFinal grouped rows: {len(grouped)}")
grouped.head(10)

---
## 10. Known Gaps & Open Questions

| Item | Status | Action needed |
|---|---|---|
| Revenue total | ✅ Verified ($135.7M) | None |
| Revenue by service line | ✅ Verified | None |
| Top clients | ✅ Verified | None |
| COS | ⚠️ Built, not cross-checked | Cross-check with John |
| People costs | ❌ Suspect ($68M vs $31M COS) | Confirm correct source with John |
| Gross margin | ❌ Dependent on people costs | Fix people costs first |
| Pipeline | ⚠️ Built, no Excel equivalent | John to confirm pipeline view |
| Targets | ❌ Only Q1 2025+ in DB | Ask if 2026 budget can be loaded |
| Actual vs Budget | ❌ Not built | Blocked on budget data |
| BE / AE Synd / RTA allocations | ❌ Not built | Need fixed cost values from Erik |
| Pendings (booked not posted) | ❌ Not in DB | Ask Paul if these should be included |
| GRR / NRR retention metrics | ❌ Not built | Lower priority, post-data model sign-off |

In [ ]:
sql = (Path("../src/queries") / "1_revenue_cogs_labour_by_period.sql").read_text()
df = pd.read_sql(sql, engine)
print(f"Rows: {len(df)}")


def _drop_no_client(df: pd.DataFrame) -> pd.DataFrame:
    return df[
        df["top_level_parent_customer_name"].notna() &
        (df["revenue"] > 0) &
        (df['sub_service_line_name'] == 'Brand Effect')
    ]

df = _drop_no_client(df)

df.head(10)




In [ ]:
# Verify Brand Effect rows — group by period and client, check counts
# Expecting count = 1 per group (one row per period/client combo)
# If count > 1 something is wrong with the grouping in the SQL

df_check = (
    df
    .groupby(["accounting_period_name", "top_level_parent_customer_name"])
    .agg(
        row_count=("revenue", "count"),
        total_revenue=("revenue", "sum")
    )
    .reset_index()
    .sort_values("row_count", ascending=False)
)

print(f"Max count per group: {df_check['row_count'].max()}")
print(f"Any count > 1: {(df_check['row_count'] > 1).any()}")
print(df_check.head(10))

In [ ]:
# Inspect the Disney Oct 2022 duplicate — find out why count = 2
# i.e. what's different between the two rows

df[
    (df["accounting_period_name"] == "Oct 2022") &
    (df["top_level_parent_customer_name"] == "Disney")
]

In [ ]:
sql = (Path("../src/queries") / "2_be_allocation.sql").read_text()
df_be = pd.read_sql(sql, engine)
print(f"Rows: {len(df_be)}")

# Sanity check — sum of monthly allocations per year should equal 1000
print("\nAllocation sum per year (should equal 1000):")
print(df_be.groupby("yr")["be_allocation_per_month"].sum().round(2))

df_be.head(10)

In [ ]:
sql = (Path("../src/queries") / "2_be_allocation.sql").read_text()
df_be = pd.read_sql(sql, engine)
print(f"Rows: {len(df_be)}")

print("\nAllocation sum per year (should equal 1000):")
print(df_be.groupby("yr")["be_allocation_per_month"].sum().round(2))

df_be.head(10)

In [ ]:
# Disney BE allocation example — walk through the maths row by row

# Step 1: Disney's annual BE revenue per year
disney_annual = df_be[
    df_be["top_level_parent_customer_name"] == "Disney"
][["yr", "vertical_name", "annual_be_rev", "annual_tot_be_rev",
   "fixed_cost_be", "be_allocation_per_month"]].drop_duplicates(
    subset=["yr", "vertical_name"]
).sort_values(["yr", "vertical_name"])

print("=== Disney Annual BE Revenue & Allocation ===")
print(disney_annual.to_string(index=False))

# Step 2: show the maths explicitly for one year
yr = 2025
row = disney_annual[disney_annual["yr"] == yr].iloc[0]
print(f"\n=== Manual check for Disney {yr} ===")
print(f"Disney annual BE rev:    ${row['annual_be_rev']:,.2f}")
print(f"Total BE rev all clients:${row['annual_tot_be_rev']:,.2f}")
print(f"Disney share:            {row['annual_be_rev']/row['annual_tot_be_rev']*100:.2f}%")
print(f"Fixed cost:              ${row['fixed_cost_be']:,.0f}")
print(f"Annual allocation:       ${row['annual_be_rev']/row['annual_tot_be_rev']*row['fixed_cost_be']:.2f}")
print(f"Monthly allocation (/12):${row['be_allocation_per_month']:.2f}")

# Step 3: confirm Disney appears in all 12 months for that year
disney_months = df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == yr)
][["accounting_period_name", "be_allocation_per_month"]]
print(f"\n=== Disney {yr} — all {len(disney_months)} months ===")
print(disney_months.to_string(index=False))
print(f"Sum of monthly allocations: ${disney_months['be_allocation_per_month'].sum():.2f}")

In [ ]:
# Check Disney 2025 sum by vertical — each should equal annual allocation
disney_2025 = df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == 2025)
]

print("Disney 2025 — sum by vertical (should match annual allocation * 12 / 12 = monthly * 12):")
print(
    disney_2025
    .groupby("vertical_name")
    .agg(
        monthly_rate=("be_allocation_per_month", "first"),
        total_allocated=("be_allocation_per_month", "sum"),
        months=("be_allocation_per_month", "count")
    )
)

print(f"\nGrand total allocated to Disney 2025: ${disney_2025['be_allocation_per_month'].sum():.2f}")
print(f"Disney's share of total annual BE cost: ${disney_2025['be_allocation_per_month'].sum() * 12 / 12:.2f}")

In [ ]:
# Show all distinct combinations for Disney TV 2025
df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == 2025) &
    (df_be["vertical_name"] == "TV")
][["accounting_period_name", "vertical_name", "service_line_name",
   "sub_service_line_name", "be_allocation_per_month"]].head(24)

In [ ]:
# Check how many distinct customer_ids map to Disney
pd.read_sql("""
    SELECT
        c.top_level_parent_customer_name,
        r.customer_id,
        c.customer_full_name,
        -SUM(r.amount) AS annual_be_rev
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.account_type_id = 'Income'
      AND r.sub_service_line_name = 'Brand Effect'
      AND r.accounting_period_is_posted = TRUE
      AND c.top_level_parent_customer_name = 'Disney'
      AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
    GROUP BY 1, 2, 3
    ORDER BY 4 DESC
""", engine)

In [ ]:
sql = (Path("../src/queries") / "2_be_allocation.sql").read_text()
df_be = pd.read_sql(sql, engine)

print("Allocation sum per year (should equal 1000):")
print(df_be.groupby("yr")["be_allocation_per_month"].sum().round(2))

# Disney 2025 check — should now show exactly 12 rows per vertical
disney_2025 = df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == 2025)
]
print(f"\nDisney 2025 rows: {len(disney_2025)} (expect 12 per vertical)")
print(disney_2025.groupby("vertical_name")["be_allocation_per_month"].agg(["count", "sum"]))

In [ ]:
df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") ]

In [ ]:
sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)
print(f"Rows: {len(df_gm)}")

# Sanity check vs Excel — 2025 totals
totals = df_gm[pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025].agg({
    "revenue": "sum",
    "cogs": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum",
    "gross_margin": "sum"
}).round(0)

print("\n2025 totals:")
print(totals)
print(f"\nGM %: {totals['gross_margin']/totals['revenue']*100:.1f}%")

df_gm.head(10)

In [ ]:
totals_2025 = df_gm[
    pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "labour": "sum"
}).round(0)

print(totals_2025)

In [ ]:
# Quick check — does using amount vs amount_usd make a difference?
pd.read_sql("""
    SELECT
        -SUM(amount)     AS total_amount,
        -SUM(amount_usd) AS total_amount_usd
    FROM core.rpt_project_revenue_and_costs
    WHERE account_type_id = 'Income'
    AND accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)

In [ ]:
pd.read_sql("""
    SELECT
        SUM(CASE WHEN account_type_id = 'COGS' THEN amount ELSE 0 END) AS cogs_only,
        SUM(CASE WHEN account_type_id IS NULL   THEN amount ELSE 0 END) AS labour_only,
        SUM(CASE WHEN account_type_id IN ('COGS') OR account_type_id IS NULL THEN amount ELSE 0 END) AS cogs_plus_labour
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)

In [ ]:
# How many rows are being dropped by the customer join?
pd.read_sql("""
    SELECT
        CASE WHEN c.top_level_parent_customer_name IS NULL
             THEN 'No customer match'
             ELSE 'Has customer' END AS customer_status,
        account_type_id,
        COUNT(*) as rows,
        SUM(amount) as total_amount
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 1, 2
""", engine)

In [ ]:
sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)

totals_2025 = df_gm[
    pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "labour": "sum"
}).round(0)

print(totals_2025)

In [ ]:

sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)

df_gm_be = df_gm[
    (pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025) &
    (df_gm["sub_service_line_name"] == "Brand Effect")
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "gross_margin": "sum",
    "labour": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum"
}).round(0)

print(df_gm_be)
print(f"GM%: {df_gm_be['gross_margin']/df_gm_be['revenue']*100:.1f}%")

In [ ]:
# Check what the be_allocation sums to across ALL rows for 2025
# not just Brand Effect rows
df_gm_2025 = df_gm[pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025]

print("be_allocation sum across ALL 2025 rows:")
print(df_gm_2025["be_allocation"].sum().round(0))

print("\nbe_allocation sum for Brand Effect rows only:")
print(df_gm_2025[df_gm_2025["sub_service_line_name"] == "Brand Effect"]["be_allocation"].sum().round(0))

print("\nbe_allocation sum for non-Brand Effect rows:")
print(df_gm_2025[df_gm_2025["sub_service_line_name"] != "Brand Effect"]["be_allocation"].sum().round(0))

In [ ]:

sql = (Path("../src/queries") / "5_gross_margin.sql").read_text()
df_gm = pd.read_sql(sql, engine)

df_gm_be = df_gm[
    (pd.to_datetime(df_gm["accounting_period_start_date"]).dt.year == 2025) &
    (df_gm["top_level_parent_customer_name"] == "Chase")
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "gross_margin": "sum",
    "labour": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum"

}).round(0)

print(df_gm_be)
print(f"GM%: {df_gm_be['gross_margin']/df_gm_be['revenue']*100:.1f}%")

In [ ]:
# What does the raw be_alloc CTE produce for 2025?
pd.read_sql("""
    WITH fixed_costs AS (
        SELECT 2025 AS yr, 10158000 AS fixed_cost_be
    ),
    be_rev AS (
        SELECT
            EXTRACT(YEAR FROM r.accounting_period_start_date)::int AS yr,
            COALESCE(c.top_level_parent_customer_name, 'Unassigned') AS top_level_parent_customer_name,
            r.vertical_name, r.service_line_name, r.sub_service_line_name,
            -SUM(r.amount) AS annual_be_rev
        FROM core.rpt_project_revenue_and_costs r
        LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
        WHERE r.account_type_id = 'Income'
          AND r.sub_service_line_name = 'Brand Effect'
          AND r.accounting_period_is_posted = TRUE
          AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
        GROUP BY 1,2,3,4,5
    ),
    tot AS (SELECT yr, SUM(annual_be_rev) AS tot FROM be_rev GROUP BY 1),
    periods AS (
        SELECT DISTINCT
            EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
            accounting_period_name
        FROM core.rpt_project_revenue_and_costs
        WHERE accounting_period_is_posted = TRUE
          AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    )
    SELECT
        SUM((b.annual_be_rev / NULLIF(t.tot,0)) * f.fixed_cost_be / 12) AS total_be_alloc
    FROM be_rev b
    JOIN tot t ON b.yr = t.yr
    JOIN fixed_costs f ON b.yr = f.yr
    CROSS JOIN periods p
""", engine)

In [ ]:
# Check dim_customers for a 'have' or similar flag column
pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core'
    AND table_name = 'dim_customers'
    ORDER BY ordinal_position
""", engine)

In [21]:
import pandas as pd
from sqlalchemy import text
from src.models.financials import get_gross_margin
from src.db.connection import get_engine

engine = get_engine()

# ── 1. load via the model function ──────────────────────────────────────────
df = get_gross_margin(year_from=2025)
df = df[df["accounting_period_start_date"].dt.year == 2025]

print("=== TOTAL revenue (all rows) ===")
print(f"  ${df['revenue'].sum()/1e6:.2f}M")

# ── 2. revenue by service line ───────────────────────────────────────────────
print("\n=== Revenue by service_line_name ===")
by_sl = df.groupby("service_line_name", dropna=False)["revenue"].sum().reset_index()
by_sl["revenue_m"] = by_sl["revenue"] / 1e6
print(by_sl.sort_values("revenue", ascending=False).to_string(index=False))
print(f"\n  Sub-total (named SLs only): ${by_sl[by_sl['service_line_name'] != '(blank)']['revenue'].sum()/1e6:.2f}M")
print(f"  Grand total incl blank:     ${by_sl['revenue'].sum()/1e6:.2f}M")

# ── 3. check row counts — are rows being duplicated? ────────────────────────
print("\n=== Row count check ===")
print(f"  Total rows in df:           {len(df):,}")
print(f"  Distinct accounting periods: {df['accounting_period_start_date'].nunique()}")

# ── 4. check for duplicates on natural key ───────────────────────────────────
key_cols = ["accounting_period_start_date", "service_line_name", "sub_service_line_name",
            "vertical_name", "top_level_parent_customer_name"]
dupes = df[df.duplicated(key_cols, keep=False)]
print(f"\n=== Duplicate rows on natural key: {len(dupes):,} ===")
if len(dupes) > 0:
    print(dupes[key_cols + ["revenue"]].head(10).to_string(index=False))

# ── 5. raw SQL total — bypass model, hit table directly ─────────────────────
print("\n=== Raw SQL revenue total (2025, income rows only) ===")
raw_sql = """
SELECT
    SUM(CASE WHEN account_type_id = 'Income' THEN -amount ELSE 0 END) AS raw_revenue
FROM core.rpt_project_revenue_and_costs
WHERE accounting_period_is_posted = TRUE
  AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
"""
with engine.connect() as conn:
    raw = pd.read_sql(text(raw_sql), conn)
print(f"  ${raw['raw_revenue'].iloc[0]/1e6:.2f}M")

# ── 6. raw SQL grouped by service line ──────────────────────────────────────
print("\n=== Raw SQL revenue by service line (2025) ===")
raw_sl_sql = """
SELECT
    service_line_name,
    SUM(CASE WHEN account_type_id = 'Income' THEN -amount ELSE 0 END) AS revenue,
    COUNT(*) AS row_count
FROM core.rpt_project_revenue_and_costs r
LEFT JOIN core.dim_customers c USING (customer_id)
WHERE accounting_period_is_posted = TRUE
  AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
GROUP BY service_line_name
ORDER BY revenue DESC
"""
with engine.connect() as conn:
    raw_sl = pd.read_sql(text(raw_sl_sql), conn)
print(raw_sl.to_string(index=False))
print(f"\n  Total: ${raw_sl['revenue'].sum()/1e6:.2f}M")

=== TOTAL revenue (all rows) ===
  $135.70M

=== Revenue by service_line_name ===
service_line_name     revenue  revenue_m
     Ad Solutions 83118879.11  83.118879
           Custom 24618215.50  24.618216
   Brand Tracking 23498047.49  23.498047
          Content  3133214.10   3.133214
     Data Science  2161222.08   2.161222
              NaN  -834067.93  -0.834068

  Sub-total (named SLs only): $135.70M
  Grand total incl blank:     $135.70M

=== Row count check ===
  Total rows in df:           5,143
  Distinct accounting periods: 12

=== Duplicate rows on natural key: 16 ===
accounting_period_start_date service_line_name sub_service_line_name vertical_name top_level_parent_customer_name  revenue
                  2025-05-01      Ad Solutions             Ad Effect           NaN                       T-Mobile -18700.0
                  2025-05-01      Ad Solutions             Ad Effect           NaN                       T-Mobile      0.0
                  2025-06-01      Ad Solution

In [22]:
# ── Find what's in SQL $135.7M that Excel's $105M excludes ─────────────────

print("=== Revenue by vertical_name ===")
by_vert = (df.groupby("vertical_name", dropna=False)["revenue"]
           .sum()
           .reset_index()
           .sort_values("revenue", ascending=False))
by_vert["revenue_m"] = by_vert["revenue"] / 1e6
print(by_vert.to_string(index=False))

print("\n=== Revenue by top_level_parent_customer_name (top 20) ===")
by_client = (df.groupby("top_level_parent_customer_name", dropna=False)["revenue"]
             .sum()
             .reset_index()
             .sort_values("revenue", ascending=False)
             .head(20))
by_client["revenue_m"] = by_client["revenue"] / 1e6
print(by_client.to_string(index=False))

print("\n=== Revenue by service_line + vertical (to find the $30M bucket) ===")
by_sl_vert = (df.groupby(["service_line_name", "vertical_name"], dropna=False)["revenue"]
              .sum()
              .reset_index()
              .sort_values("revenue", ascending=False))
by_sl_vert["revenue_m"] = by_sl_vert["revenue"] / 1e6
print(by_sl_vert.to_string(index=False))

# ── T-Mobile dupe investigation ─────────────────────────────────────────────
print("\n=== T-Mobile Ad Effect detail (the 16 dupe rows) ===")
tmobile = df[
    (df["top_level_parent_customer_name"] == "T-Mobile") &
    (df["sub_service_line_name"] == "Ad Effect")
][["accounting_period_start_date", "vertical_name", "revenue", "cogs"]].sort_values("accounting_period_start_date")
print(tmobile.to_string(index=False))
print(f"\n  T-Mobile Ad Effect total revenue: ${tmobile['revenue'].sum()/1e3:.1f}k")

=== Revenue by vertical_name ===
       vertical_name     revenue     revenue_m
   Theatrical/Studio 32000579.52  3.200058e+01
  Financial Services 17096358.55  1.709636e+01
            Platform 15437168.65  1.543717e+01
                Auto 11004312.94  1.100431e+01
                 CPG 10847103.13  1.084710e+01
       Telco & Cable 10246337.79  1.024634e+01
                  TV  9672539.53  9.672540e+00
           Streaming  7828599.96  7.828600e+00
   Retail-Restaurant  7755679.41  7.755679e+00
               Other  5084923.44  5.084923e+00
Sports & Live Events  2261861.61  2.261862e+00
    Music & Podcasts  1471663.44  1.471663e+00
              Gaming  1176809.00  1.176809e+00
              Health  1156782.95  1.156783e+00
              Travel  1058468.68  1.058469e+00
                Tech   927208.85  9.272088e-01
                 M&E   372420.02  3.724200e-01
          Publishers   197087.50  1.970875e-01
              Agency    66000.00  6.600000e-02
        Social Media    411

In [23]:
# ── Find where the 135 vs 105 gap lives ─────────────────────────────────────

print("=== Revenue by sub_service_line_name (2025) ===")
by_ssl = (
    df.groupby(["service_line_name", "sub_service_line_name"], dropna=False)["revenue"]
    .sum()
    .reset_index()
    .sort_values("revenue", ascending=False)
)
by_ssl["revenue_m"] = (by_ssl["revenue"] / 1e6).round(2)
print(by_ssl.to_string(index=False))
print(f"\nTotal: ${by_ssl['revenue'].sum()/1e6:.2f}M")

# ── Check Ad Solutions breakdown specifically ────────────────────────────────
print("\n=== Ad Solutions sub-lines ===")
ad_sol = by_ssl[by_ssl["service_line_name"] == "Ad Solutions"]
print(ad_sol.to_string(index=False))
print(f"Ad Solutions total: ${ad_sol['revenue'].sum()/1e6:.2f}M")

# ── Check the 788k labour rows — what revenue do they actually carry? ─────────
print("\n=== NULL service_line rows breakdown ===")
null_sl = df[df["service_line_name"].isna()]
print(f"Row count:          {len(null_sl):,}")
print(f"Revenue sum:        ${null_sl['revenue'].sum()/1e6:.2f}M")
print(f"COGS sum:           ${null_sl['cogs'].sum()/1e6:.2f}M")
print(f"Labour sum:         ${null_sl['labour'].sum()/1e6:.2f}M")

print("\nSample of null SL rows:")
print(null_sl[["accounting_period_start_date", "sub_service_line_name",
               "top_level_parent_customer_name", "revenue", "labour"]].head(10).to_string(index=False))

# ── The 16 dupes — are they material? ───────────────────────────────────────
print("\n=== Dupe revenue impact ===")
key_cols = ["accounting_period_start_date", "service_line_name", "sub_service_line_name",
            "vertical_name", "top_level_parent_customer_name"]
dupes = df[df.duplicated(key_cols, keep=False)]
print(f"Revenue in dupe rows: ${dupes['revenue'].sum()/1e6:.2f}M")
print(f"Unique dupe combos:   {dupes.drop_duplicates(key_cols).shape[0]}")

# ── Compare: Excel's $105M — does it match if we exclude a vertical? ─────────
print("\n=== Revenue by vertical ===")
by_vert = df.groupby("vertical_name", dropna=False)["revenue"].sum().reset_index()
by_vert["revenue_m"] = (by_vert["revenue"] / 1e6).round(2)
print(by_vert.sort_values("revenue", ascending=False).to_string(index=False))

# ── Check if internal vertical = ~$30M gap ───────────────────────────────────
print("\n=== Revenue EXCLUDING 'Internal' vertical ===")
excl_internal = df[~df["vertical_name"].isin(["Internal", "internal", "INTERNAL"])]
print(f"${excl_internal['revenue'].sum()/1e6:.2f}M")

=== Revenue by sub_service_line_name (2025) ===
service_line_name sub_service_line_name     revenue  revenue_m
     Ad Solutions                    CA 30604372.83      30.60
           Custom                Custom 24618215.50      24.62
   Brand Tracking                 Brand 23498047.49      23.50
     Ad Solutions          Brand Effect 22873198.65      22.87
     Ad Solutions             Ad Effect 17687061.12      17.69
     Ad Solutions       Campaign Effect  5876675.50       5.88
     Ad Solutions      Ad Solutions RTA  3225532.75       3.23
          Content                Custom  3133214.10       3.13
     Ad Solutions          Data Science  2852038.26       2.85
     Data Science          Data Science  2161222.08       2.16
              NaN                   NaN  -834067.93      -0.83

Total: $135.70M

=== Ad Solutions sub-lines ===
service_line_name sub_service_line_name     revenue  revenue_m
     Ad Solutions                    CA 30604372.83      30.60
     Ad Solutions    

In [24]:
ca_rows = df[df["sub_service_line_name"] == "CA"]
print(ca_rows[["service_line_name", "sub_service_line_name",
               "vertical_name", "top_level_parent_customer_name",
               "revenue"]].head(10).to_string(index=False))
print(f"\nvertical_name nulls: {ca_rows['vertical_name'].isna().sum()} / {len(ca_rows)}")
print(f"customer nulls:      {ca_rows['top_level_parent_customer_name'].isna().sum()} / {len(ca_rows)}")

service_line_name sub_service_line_name        vertical_name top_level_parent_customer_name   revenue
     Ad Solutions                    CA    Theatrical/Studio             Studiocanal S.A.S.  19943.14
     Ad Solutions                    CA    Theatrical/Studio                           Sony  48935.73
     Ad Solutions                    CA    Theatrical/Studio                       Skydance  17500.00
     Ad Solutions                    CA    Theatrical/Studio               Paramount Global     -0.00
     Ad Solutions                    CA    Theatrical/Studio                        Netflix  59799.00
     Ad Solutions                    CA    Theatrical/Studio                   NBCUniversal 865126.54
     Ad Solutions                    CA Sports & Live Events                            NBA  41666.67
     Ad Solutions                    CA    Theatrical/Studio                      Lionsgate  63352.41
     Ad Solutions                    CA    Theatrical/Studio          Ketchup Ente

In [29]:
ca = df[df["sub_service_line_name"] == "CA"]
print(f"CA rows in df: {len(ca)}")
print(f"CA revenue: ${ca['revenue'].sum()/1e6:.2f}M")
print(f"CA gross_margin nulls: {ca['gross_margin'].isna().sum()}")
print(ca[["revenue","cogs","gross_margin","be_allocation","ae_allocation","rta_allocation"]].sum())

CA rows in df: 233
CA revenue: $30.60M
CA gross_margin nulls: 0
revenue           30604372.83
cogs              10405289.18
gross_margin      20199083.65
be_allocation            0.00
ae_allocation            0.00
rta_allocation           0.00
dtype: float64
